# Project 12 — Local Policy Assistant

## Search HR/IT/Company Policies with Citations

**Goal:** Build a local RAG assistant that indexes company policy documents,
answers employee questions with exact section citations, and refuses to
answer when a topic is not covered — all running locally.

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

```
Policy docs (.txt) ──► DirectoryLoader ──► docs with metadata
                                                   │
                                         RecursiveTextSplitter ──► chunks
                                                                    │
                                              Ollama Embeddings ──► ChromaDB
                                                                       │
Employee question ──► embed ──► retrieval ──► top chunks ──► LLM ──► Cited Answer
```

### What You'll Learn

1. Create structured policy documents with section numbering
2. Enrich document metadata for citation support
3. Build a citation-aware RAG chain
4. Handle cross-policy questions
5. Gracefully refuse out-of-scope topics

### Prerequisites

- **Ollama** installed and running (`ollama serve`)
- Models pulled: `ollama pull nomic-embed-text` and `ollama pull qwen3:8b`
- Python 3.9+

In [ ]:
# Install dependencies (uncomment and run once)
!pip install -q langchain langchain-ollama langchain-community langchain-text-splitters chromadb

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests, sys

OLLAMA_BASE = "http://localhost:11434"

try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✅ Ollama is running — {len(models)} model(s) available")
    for m in models:
        print(f"   • {m}")
except Exception as e:
    print(f"❌ Cannot reach Ollama at {OLLAMA_BASE}: {e}")
    print("\nFix: start Ollama and run:")
    print("  ollama pull qwen3:8b")
    print("  ollama pull nomic-embed-text")

## Step 2 — Configure LLM and Embeddings

Same local-only setup: **nomic-embed-text** for vectors, **qwen3:8b** for answers.
No API keys, no cloud calls.

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

LLM_MODEL = "qwen3:8b"
EMBED_MODEL = "nomic-embed-text"

llm = ChatOllama(model=LLM_MODEL, temperature=0)
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

vec = embeddings.embed_query("test")
print(f"✅ Embedding model ready — dimension: {len(vec)}")

resp = llm.invoke("Say 'hello' in one word.")
print(f"✅ LLM ready — response: {resp.content[:120]}")

## Step 3 — Create Sample Policy Documents

We create **4 realistic company policies**, each with numbered sections.
Section numbers are critical: the LLM cites the exact section
(e.g., "Remote Work Policy, Section 2"). In production these would be
real HR/IT PDFs converted to text.

In [ ]:
from pathlib import Path

policies_dir = Path("sample_data/policies")
policies_dir.mkdir(parents=True, exist_ok=True)

policies = {
    "remote_work_policy.txt": (
        "Remote Work Policy — Effective January 2025\n\n"
        "Section 1: Eligibility\n"
        "All full-time employees who have completed 90 days of employment are eligible "
        "for remote work. Contractors and part-time employees require explicit written "
        "approval from their direct manager and HR.\n\n"
        "Section 2: Equipment\n"
        "The company provides a laptop, external monitor, and a $500 one-time home "
        "office stipend for ergonomic equipment. Equipment must be returned within "
        "14 days of separation from the company.\n\n"
        "Section 3: Working Hours\n"
        "Core hours are 10:00 AM to 3:00 PM in your local timezone. All employees "
        "must be available on Slack during core hours. Total weekly hours: 40. "
        "Overtime requires pre-approval from your manager.\n\n"
        "Section 4: Security Requirements\n"
        "All work must be done on company-approved devices only. VPN must be active "
        "when accessing any internal systems. Public WiFi is prohibited without VPN. "
        "Screen lock must be set to 2 minutes of inactivity."
    ),
    "pto_policy.txt": (
        "Paid Time Off (PTO) Policy — Effective January 2025\n\n"
        "Section 1: Accrual\n"
        "Full-time employees accrue 1.67 days per month (20 days per year). "
        "Maximum accrual cap: 30 days. Unused PTO does not roll over beyond the cap.\n\n"
        "Section 2: Requesting PTO\n"
        "Submit requests via the HR Portal at least 5 business days in advance. "
        "Requests of 5 or more consecutive days require VP-level approval.\n\n"
        "Section 3: Holidays\n"
        "The company observes 10 federal holidays plus 2 floating holidays. "
        "Floating holidays must be used within the calendar year.\n\n"
        "Section 4: Sick Leave\n"
        "Employees receive 10 sick days per year, separate from PTO. "
        "A doctor's note is required for absences exceeding 3 consecutive days.\n\n"
        "Section 5: Bereavement Leave\n"
        "5 days for immediate family (spouse, child, parent), "
        "3 days for extended family."
    ),
    "expense_policy.txt": (
        "Expense Reimbursement Policy — Effective January 2025\n\n"
        "Section 1: Eligible Expenses\n"
        "Business travel, client meals, conference registration fees, and "
        "work-related software subscriptions are eligible for reimbursement.\n\n"
        "Section 2: Spending Limits\n"
        "Meals: $50/person for client dinners, $25 for solo business meals. "
        "Hotels: Up to $250/night in major metros (NYC, SF, LA), $175 elsewhere. "
        "Flights: Economy class for domestic. Business class for international 6+ hours.\n\n"
        "Section 3: Submission Process\n"
        "Submit all expenses within 30 calendar days via Concur. Receipts required "
        "for all expenses over $25. Late submissions (over 60 days) will be denied.\n\n"
        "Section 4: Approval Hierarchy\n"
        "Under $500: Manager approval. $500-$5,000: Director approval. "
        "Over $5,000: VP approval. International travel requires pre-approval.\n\n"
        "Section 5: Non-Reimbursable Items\n"
        "Personal meals, alcohol (unless client entertainment), personal travel "
        "extensions, premium economy upgrades (domestic), gym memberships."
    ),
    "it_security_policy.txt": (
        "IT Security Policy — Effective January 2025\n\n"
        "Section 1: Password Requirements\n"
        "Minimum 14 characters with uppercase, lowercase, number, and special character. "
        "Passwords must be changed every 90 days. No reuse of last 12 passwords. "
        "Multi-factor authentication (MFA) is mandatory for all accounts.\n\n"
        "Section 2: Device Management\n"
        "Only company-issued or IT-approved personal devices may access company systems. "
        "All devices must have endpoint protection. Automatic OS updates must be enabled.\n\n"
        "Section 3: Data Classification\n"
        "Confidential: Financial data, PII, trade secrets — restricted access only. "
        "Internal: Company comms, project docs — employee access only. "
        "Public: Marketing materials — no restrictions.\n\n"
        "Section 4: Incident Response\n"
        "Report all security incidents to security@company.com within 1 hour. "
        "Do not attempt to investigate on your own. Preserve evidence. "
        "IT Security responds within 30 minutes during business hours.\n\n"
        "Section 5: Software Installation\n"
        "Only IT-approved software on company devices. Submit requests via IT portal. "
        "Open-source software requires security review. No torrent software."
    ),
}

for fname, content in policies.items():
    (policies_dir / fname).write_text(content, encoding="utf-8")

print(f"✅ Created {len(policies)} policy documents:")
for fname in policies:
    print(f"   📄 {fname}")

## Step 4 — Load and Enrich Policy Documents

We use `DirectoryLoader` to load all `.txt` files, then enrich each document's
metadata with the policy name extracted from the filename. This metadata
flows through to every chunk and enables citation in the final answer.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "sample_data/policies", glob="*.txt",
    loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"},
)
docs = loader.load()

for doc in docs:
    fname = Path(doc.metadata["source"]).name
    doc.metadata["policy_name"] = fname.replace("_policy.txt", "").replace("_", " ").title()
    doc.metadata["type"] = "company_policy"

print(f"✅ Loaded {len(docs)} policy documents:")
for doc in docs:
    print(f"   [{doc.metadata['policy_name']:20s}] {len(doc.page_content):,} chars")

## Step 5 — Chunk Policies for Retrieval

Chunk with 400 characters and 50-character overlap. The splitter naturally
breaks on double-newlines (section boundaries), keeping sections mostly intact.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "],
)
chunks = splitter.split_documents(docs)

print(f"Split {len(docs)} policies → {len(chunks)} chunks\n")
for i, c in enumerate(chunks):
    policy = c.metadata.get("policy_name", "?")
    print(f"  Chunk {i:2d} | [{policy:20s}] {len(c.page_content):3d} chars | {c.page_content[:60]}...")

## Step 6 — Build the Vector Store

In [ ]:
from langchain_community.vectorstores import Chroma
import shutil

CHROMA_DIR = "sample_data/policy_chroma"

if Path(CHROMA_DIR).exists():
    shutil.rmtree(CHROMA_DIR)

vectorstore = Chroma.from_documents(
    chunks, embeddings,
    persist_directory=CHROMA_DIR, collection_name="policies",
)
print(f"✅ Vector store created — {vectorstore._collection.count()} policy chunks indexed")

## Step 7 — Test Retrieval Quality

Verify the retriever finds the correct policy sections before building the QA chain.

In [ ]:
test_queries = [
    "How many PTO days do I get?",
    "What is the password policy?",
    "Can I use my personal laptop?",
    "What are the hotel expense limits?",
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = vectorstore.similarity_search_with_score(query, k=2)
    for i, (doc, score) in enumerate(results):
        policy = doc.metadata.get("policy_name", "?")
        print(f"   [{i+1}] score={score:.4f} | {policy:20s} | {doc.page_content[:75]}...")

## Step 8 — Build the Citation-Aware Policy Chain

The system prompt instructs the LLM to:
1. Answer ONLY from the provided policy excerpts
2. Always cite the policy name and section number
3. Refuse to answer if the topic is not covered

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

system_prompt = (
    "You are the company's HR/IT policy assistant. Answer employee questions "
    "using ONLY the policy documents provided below.\n\n"
    "Rules:\n"
    "1. Always cite the specific policy name and section number "
    "(e.g., 'Per the PTO Policy, Section 2...').\n"
    "2. If the answer spans multiple policies, cite each one.\n"
    "3. If the topic is not covered by any policy, say: "
    "'This is not covered in our current policies. "
    "Please contact HR at hr@company.com for guidance.'\n"
    "4. Be precise — stay close to policy language.\n\n"
    "Policy Excerpts:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


def policy_query(question: str) -> dict:
    """Query the policy assistant with citations."""
    docs = retriever.invoke(question)
    context_text = "\n\n---\n\n".join(
        f"[{d.metadata.get('policy_name', '?')} Policy]\n{d.page_content}" for d in docs
    )
    response = (prompt | llm | StrOutputParser()).invoke(
        {"context": context_text, "input": question}
    )
    return {"answer": response, "sources": docs}


print("✅ Policy assistant chain ready!")

## Step 9 — Test the Policy Assistant

Let's ask employee questions spanning different policies and verify citations.

In [ ]:
questions = [
    "How many PTO days do I get per year?",
    "What's the hotel reimbursement limit in New York?",
    "Am I eligible for remote work as a contractor?",
    "Do I need approval for a $600 expense?",
    "What happens to unused PTO?",
    "Can I use public WiFi when working remotely?",
    "What is the password length requirement?",
    "How do I report a security incident?",
]

for q in questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    result = policy_query(q)
    print(f"\nA: {result['answer']}")
    policies_cited = set(s.metadata.get('policy_name', '?') for s in result['sources'])
    print(f"\n📋 Policies referenced: {policies_cited}")

## Step 10 — Test Out-of-Scope Questions

The assistant should gracefully decline topics not covered by any policy.

In [ ]:
out_of_scope = [
    "What is the company's stock price?",
    "Can I bring my dog to the office?",
    "What is the dress code?",
    "Who is the CEO?",
]

print("=== Out-of-Scope Questions (should decline) ===\n")
for q in out_of_scope:
    result = policy_query(q)
    answer = result["answer"]
    print(f"Q: {q}")
    print(f"A: {answer}")
    grounded = any(phrase in answer.lower() for phrase in [
        "not covered", "contact hr", "no policy", "not addressed",
        "not mentioned", "not in our", "don't have",
    ])
    status = "✅ Yes" if grounded else "⚠️  No"
    print(f"   Grounded refusal: {status}\n")

## Step 11 — Cross-Policy Questions

Some questions require information from multiple policies.

In [ ]:
cross_questions = [
    "What security rules apply when working remotely?",
    "I need to travel internationally — what about expenses and security?",
]

for q in cross_questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    result = policy_query(q)
    print(f"\nA: {result['answer']}")
    policies_cited = set(s.metadata.get('policy_name', '?') for s in result['sources'])
    print(f"\n📋 Policies referenced: {policies_cited}")

## Step 12 — Interactive Policy Helper

In [ ]:
def ask(question: str) -> str:
    """Ask the policy assistant and display cited answer."""
    result = policy_query(question)
    print(f"Answer:\n{result['answer']}\n")
    print(f"Sources ({len(result['sources'])} chunks):")
    for i, doc in enumerate(result['sources']):
        policy = doc.metadata.get('policy_name', '?')
        print(f"   [{i+1}] {policy}: {doc.page_content[:80]}...")
    return result['answer']

_ = ask("How long do I have to submit an expense report?")

## Limitations & Tradeoffs

| Aspect | What happens | How to improve |
|--------|-------------|----------------|
| **Section detection** | Section numbers in text, not structured | Parse into separate chunks with metadata |
| **Policy updates** | Static docs need re-indexing | Add versioning pipeline |
| **Ambiguity** | Fuzzy/overlapping rules | Add clarification step |
| **Multi-turn** | Each question independent | Add conversation memory |
| **Access control** | All policies visible | Add role-based filtering |

## What You Learned

1. **Multi-document indexing** — loading a directory of policy files with metadata
2. **Metadata enrichment** — adding policy names for citation support
3. **Citation-aware prompting** — instructing the LLM to cite policy + section
4. **Cross-policy retrieval** — answering questions that span multiple documents
5. **Graceful refusal** — declining topics not covered by any policy

## Exercises

1. **Add a new policy** — create an `it_acceptable_use_policy.txt` and test
2. **Section-level metadata** — parse sections into separate docs
3. **Policy comparison** — ask the LLM to compare rules across policies
4. **Confidence scoring** — show retrieval scores to users
5. **Escalation path** — suggest contacting HR for ambiguous questions

---

*Next project: **13 — Local Multi-PDF Research Librarian***